# Tarea 2 Procesamiento Masivo de Datos COM4002
- Integrantes: Martin Navarro Toro.
- Docente encargado: Alex Di Genova B.


# Problema 1

## Parte 1

In [ ]:
# instalamos ambas librerias MPI y OpenMP
!apt install mpich libomp-dev

Mejora con lo solicitado por el profesor

In [ ]:
%%writefile OpenMP.c
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>
#include <omp.h>

#define vector10M 10000000   // 10M
#define vector100M 100000000 // 100M
#define vector1B 1000000000 // 1000M

void initialize_vectors(char* vectorA, char* vectorB, int size) {
    #pragma omp parallel for
    for (int i = 0; i < size; i++) {
        vectorA[i] = (char)(rand() % 255);
        vectorB[i] = (char)(rand() % 255);
    }
}

int main(int argc, char* argv[]) {
    int rank, size, vector_size;
    int local_size;
    char *vectorA = NULL, *vectorB = NULL;  // vector global
    char *local_vectorA, *local_vectorB;  // vector local
    double local_sum = 0.0, global_sum = 0.0;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    // seleccionar tamaño del vector manualmente (entre vector10M o vector100M y vector1B)
    vector_size = vector1B; // escribir el tamaño del vector en este apartado
    local_size = vector_size / size;

    if (rank == 0) {
        vectorA = malloc(vector_size * sizeof(char));
        vectorB = malloc(vector_size * sizeof(char));

        if (vectorA == NULL || vectorB == NULL) {
            fprintf(stderr, "Error al asignar memoria para los vectores globales.\n");
            MPI_Abort(MPI_COMM_WORLD, EXIT_FAILURE);
        }

        printf("Inicializando vectores en el proceso 0...\n");
        initialize_vectors(vectorA, vectorB, vector_size);
    }

    // asignacion de memoria para los vectores locales en cada proceso
    local_vectorA = malloc(local_size * sizeof(char));
    local_vectorB = malloc(local_size * sizeof(char));

    if (local_vectorA == NULL || local_vectorB == NULL) {
        fprintf(stderr, "Error al asignar memoria para los vectores locales en el proceso %d.\n", rank);
        MPI_Abort(MPI_COMM_WORLD, EXIT_FAILURE);
    }

    MPI_Scatter(vectorA, local_size, MPI_CHAR, local_vectorA, local_size, MPI_CHAR, 0, MPI_COMM_WORLD);
    MPI_Scatter(vectorB, local_size, MPI_CHAR, local_vectorB, local_size, MPI_CHAR, 0, MPI_COMM_WORLD);

    // calculo del producto escalar local utilizando OpenMP
    local_sum = 0.0;
    #pragma omp parallel for reduction(+:local_sum)
    for (int i = 0; i < local_size; i++) {
        local_sum += (double)((int)local_vectorA[i] * (int)local_vectorB[i]);
    }

    printf("Proceso %d: Producto escalar local = %lf\n", rank, local_sum);

    MPI_Reduce(&local_sum, &global_sum, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("Producto escalar global para tamaño %d: %lf\n", vector_size, global_sum);
    }

    free(local_vectorA);
    free(local_vectorB);
    if (rank == 0) {
        free(vectorA);
        free(vectorB);
    }

    // finalizar MPI
    MPI_Finalize();

    return 0;
}

Writing OpenMP.c


In [ ]:
!mpicc -o openmp OpenMP.c  -fopenmp

In [ ]:
%env OMP_NUM_THREADS=4
!mpirun --oversubscribe --allow-run-as-root -np 4 ./openmp

env: OMP_NUM_THREADS=4
Inicializando vectores en el proceso 0...
Proceso 1: Producto escalar local = 837798.000000
Proceso 2: Producto escalar local = 81102931.000000
Proceso 0: Producto escalar local = -45437581.000000
Proceso 3: Producto escalar local = 68111821.000000
Producto escalar global para tamaño 1000000000: 104614969.000000


## Parte 2

In [ ]:
%%writefile hilos.c
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>
#include <pthread.h>

#define vector10M 10000000   // 10M
#define vector100M 100000000 // 100M
#define vector1B 1000000000 // 1000M
#define NUM_THREADS_PER_PROCESS 4

// almacenamiento de cada hilo
struct ThreadData {
    char* vectorA;
    char* vectorB;
    int local_size;
    int local_start;
    double local_sum;
};

void* calculateDotProduct(void* arg) {
    struct ThreadData* data = (struct ThreadData*)arg;
    data->local_sum = 0.0;
    for (int i = data->local_start; i < data->local_start + data->local_size; i++) {
        data->local_sum += (double)((int)data->vectorA[i] * (int)data->vectorB[i]);
    }
    return NULL;
}

int main(int argc, char* argv[]) {
    int rank, size, vector_size;
    int local_size, global_size_per_thread;
    char *vectorA = NULL, *vectorB = NULL; // vector global
    char *local_vectorA, *local_vectorB;  // vector local para cada proceso
    pthread_t threads[NUM_THREADS_PER_PROCESS];
    struct ThreadData threadData[NUM_THREADS_PER_PROCESS];
    double local_sum = 0.0, global_sum = 0.0;

    // inicializar MPI
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    // seleccionar el tamaño del vector manualmente (entre vector10M, vector100M y vector1B)
    vector_size = vector1B;
    global_size_per_thread = vector_size / (size * NUM_THREADS_PER_PROCESS);
    local_size = global_size_per_thread * NUM_THREADS_PER_PROCESS;

    if (rank == 0) {
        vectorA = malloc(vector_size * sizeof(char));
        vectorB = malloc(vector_size * sizeof(char));

        if (vectorA == NULL || vectorB == NULL) {
            fprintf(stderr, "Error al asignar memoria para los vectores globales.\n");
            MPI_Abort(MPI_COMM_WORLD, EXIT_FAILURE);
        }

        printf("Inicializando vectores en el proceso 0...\n");
        for (int i = 0; i < vector_size; i++) {
            vectorA[i] = (char)(rand() % 255);
            vectorB[i] = (char)(rand() % 255);
        }
    }

    local_vectorA = malloc(local_size * sizeof(char));
    local_vectorB = malloc(local_size * sizeof(char));

    if (local_vectorA == NULL || local_vectorB == NULL) {
        fprintf(stderr, "Error al asignar memoria para los vectores locales en el proceso %d.\n", rank);
        MPI_Abort(MPI_COMM_WORLD, EXIT_FAILURE);
    }

    MPI_Scatter(vectorA, local_size, MPI_CHAR, local_vectorA, local_size, MPI_CHAR, 0, MPI_COMM_WORLD);
    MPI_Scatter(vectorB, local_size, MPI_CHAR, local_vectorB, local_size, MPI_CHAR, 0, MPI_COMM_WORLD);

    local_sum = 0.0;
    for (int i = 0; i < NUM_THREADS_PER_PROCESS; i++) {
        threadData[i].vectorA = local_vectorA;
        threadData[i].vectorB = local_vectorB;
        threadData[i].local_size = global_size_per_thread;
        threadData[i].local_start = i * global_size_per_thread;
        threadData[i].local_sum = 0.0;

        pthread_create(&threads[i], NULL, calculateDotProduct, &threadData[i]);
    }

    for (int i = 0; i < NUM_THREADS_PER_PROCESS; i++) {
        pthread_join(threads[i], NULL);
        local_sum += threadData[i].local_sum;
    }

    MPI_Reduce(&local_sum, &global_sum, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("Producto escalar global para tamaño %d: %lf\n", vector_size, global_sum);
    }

    free(local_vectorA);
    free(local_vectorB);
    if (rank == 0) {
        free(vectorA);
        free(vectorB);
    }

    // finalizar MPI
    MPI_Finalize();

    return 0;
}

Overwriting hilos.c


In [ ]:
!mpicc -o hilos hilos.c -lpthread

In [ ]:
!mpirun --oversubscribe --allow-run-as-root -np 4 ./hilos

# Problema 2

In [ ]:
# configuracion pre-procesamiento necesaria
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

!wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.1/hadoop-3.3.1.tar.gz
!tar -xzf hadoop-3.3.1.tar.gz
!mv hadoop-3.3.1 hadoop

# configuracion de la variable entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop"
os.environ["PATH"] += ":/content/hadoop/bin:/content/hadoop/sbin"

# configura un Hadoop para ejecucion local
!echo -e "\nexport JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64\nexport HADOOP_HOME=/content/hadoop\nexport PATH=$PATH:$HADOOP_HOME/bin" >> ~/.bashrc
!source ~/.bashrc


In [ ]:
# descargar primer archivo
!wget -q -O ecoli-k12-ref.fna https://raw.githubusercontent.com/adigenova/wengan_demo/master/ecoli/reference/ecoli-k12-ref.fna

# crea una carpeta en HDFS y sube el archivo
!hadoop fs -mkdir -p /input
!hadoop fs -put ecoli-k12-ref.fna /input/


In [ ]:
%%writefile mapper.py
import sys

def read_fasta(file):
    seq = ""
    for line in file:
        line = line.strip()
        if not line or line.startswith(">"):
            if seq:
                yield seq
            seq = ""
        else:
            seq += line
    if seq:
        yield seq

k = 15 # tamaño de los k-mer
for sequence in read_fasta(sys.stdin):
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i + k]
        print(f"{kmer}\t1")

Writing mapper.py


In [ ]:
%%writefile reducer.py
import sys

current_kmer = None
current_count = 0

for line in sys.stdin:
    kmer, count = line.strip().split("\t")
    count = int(count)

    if current_kmer == kmer:
        current_count += count
    else:
        if current_kmer:
            print(f"{current_kmer}\t{current_count}")
        current_kmer = kmer
        current_count = count

if current_kmer:
    print(f"{current_kmer}\t{current_count}")


Overwriting reducer.py


In [ ]:
# ejecucion del trabaja de MapReduce en Hadoop (primero)
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar \
    -input /input/ecoli-k12-ref.fna \
    -output /output/kmer_count \
    -mapper "python3 mapper.py" \
    -reducer "python3 reducer.py"

In [ ]:
# utilizamos esta parte para visualizar los datos
!hadoop fs -cat /output/kmer_count/part-00000

In [ ]:
# descargar segundo archivo
!wget -q -O EC.50X.R1.fastq.gz "https://github.com/adigenova/wengan_demo/raw/master/ecoli/reads/EC.50X.R1.fastq.gz"


In [ ]:
# descomprimir el archivo
!gunzip EC.50X.R1.fastq.gz

# sube el archivo descomprimido a un HDFS
!hadoop fs -mkdir -p /input
!hadoop fs -put EC.50X.R1.fastq /input/

gzip: EC.50X.R1.fastq already exists; do you wish to overwrite (y or n)? y
put: `/input/EC.50X.R1.fastq': File exists


In [ ]:
%%writefile mapper1.py
import sys

def read_fastq(file):
    while True:
        file.readline()
        seq = file.readline().strip()
        file.readline()
        file.readline()
        if not seq:
            break
        yield seq

k = 15  # define el tamaño del k-mer
for sequence in read_fastq(sys.stdin):
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i + k]
        print(f"{kmer}\t1")


Overwriting mapper1.py


- Se dejar reduce.py de la misma forma

In [ ]:
# ejecutar el MapReduce en Hadoop
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar \
    -input /input/EC.50X.R1.fastq \
    -output /output/kmer_count_fastq \
    -mapper "python3 mapper1.py" \
    -reducer "python3 reducer.py"

2024-11-19 20:17:32,666 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2024-11-19 20:17:32,760 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2024-11-19 20:17:32,760 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2024-11-19 20:17:32,775 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2024-11-19 20:17:32,789 ERROR streaming.StreamJob: Error Launching job : Output directory file:/output/kmer_count_fastq already exists
Streaming Command Failed!


In [ ]:
# para visualizar los resultados
!hadoop fs -cat /output/kmer_count_fastq/part-00000

Parte 3

In [ ]:
// deficion de los procesos de Nextflow

process INDEX {
    input:
    path genomeFile // archivo "genoma.fa"

    output:
    path "genome.index"

    script:
    """
    # simulacion de la indexacion del genoma
    touch genome.index
    """
}

process FASTQC {
    input:
    path sample_fwd // archivo de entrada "sample.fwd.fq.gz" , archivo de ejemplo
    path sample_rev // archivo de entrada "sample.rev.fq.gz" , archivo de ejemplo

    output:
    path "${sample_fwd.baseName}_readqc.txt" // archivo de salida para la calidad de lectura

    script:
    """
    # simulacion del control de calidad de las muestras
    touch ${sample_fwd.baseName}_readqc.txt
    """
}

process ALN {
    input:
    path sample_fwd
    path sample_rev
    path genomeIndex // archivo de indice del genoma

    output:
    path "${sample_fwd.baseName}.bam" // archivo BAM resultante de la alineacion

    script:
    """
    # simulacion del proceso de alineamiento
    touch ${sample_fwd.baseName}.bam
    """
}

process QUALIMAP {
    input:
    path bamFile // archivo BAM
    path gtfFile // archivo GTF de anotacion

    output:
    path "${bamFile.baseName}_qualimap.txt" // archivo de salida qualimap por muestra

    script:
    """
    # simulacion de la generacion de un informe de qualimap
    touch ${bamFile.baseName}_qualimap.txt
    """
}

process RESUME_QUALIMAP {
    input:
    path qualimapFiles // lista de archivos de salida qualimap

    output:
    path "all_qualimap.txt" // archivo combinado de qualimap

    script:
    """
    # simulacion de la agregacion de resultados de qualimap
    cat ${qualimapFiles.join(" ")} > all_qualimap.txt
    """
}

process QUANT_GENES {
    input:
    path bamFile // archivo BAM

    output:
    path "${bamFile.baseName}_expression.txt" // archivo de salida de expresion por muestra

    script:
    """
    # simulacion de cuantificacion de genes
    touch ${bamFile.baseName}_expression.txt
    """
}

process RESUME_QUANT_GENES {
    input:
    path expressionFiles // lista de archivos de expresion de genes

    output:
    path "expression_matrix.txt" // matriz de expresion combinada

    script:
    """
    # simulacion de la creacion de una matriz de expresion
    cat ${expressionFiles.join(" ")} > expression_matrix.txt
    """
}

process CALL_VARIANTS {
    input:
    path bamFiles // lista de archivos BAM

    output:
    path "mutations.vcf" // archivo de variantes combinado

    script:
    """
    # simulacion de la llamada a variantes
    touch mutations.vcf
    """
}

// Deficion de el flujo de trabajo

workflow {
    // carga el archivo de referencia y las muestras
    genomeFile = file("genome.fa")
    gtfFile = file("genome_annotation.gtf")
    samples = Channel.from([file("sample1.fwd.fq.gz"), file("sample1.rev.fq.gz")])

    // etapa n°1: indexacion del genoma
    genomeIndex = INDEX(genomeFile)

    // etapa n°2: control de calidad FASTQC para cada muestra
    fastqcResults = samples.map { fwd, rev ->
        FASTQC(fwd, rev)
    }

    // etapa n°3: alineamiento de muestras al genoma de referencia
    bamFiles = samples.map { fwd, rev ->
        ALN(fwd, rev, genomeIndex)
    }

    // etapa n°4: generacion de informes de calidad con Qualimap
    qualimapResults = bamFiles.map { bam ->
        QUALIMAP(bam, gtfFile)
    }

    // etapa n°5: agregacion de resultados de Qualimap
    allQualimap = RESUME_QUALIMAP(qualimapResults)

    // etapa n°6: cuantificacion de genes por muestra
    geneExpressionFiles = bamFiles.map { bam ->
        QUANT_GENES(bam)
    }

    // etapa n°7: creacion de matriz de expresion
    expressionMatrix = RESUME_QUANT_GENES(geneExpressionFiles)

    // etapa n°8: Llamado a variantes
    mutationsFile = CALL_VARIANTS(bamFiles)
}

Parte 4

In [ ]:
%%capture
!git clone https://github.com/joerg84/ArangoDBUniversity.git
!rsync -av ArangoDBUniversity/ ./ --exclude=.git
!pip3 install pyarango
!pip3 install "python-arango>=5.0"

In [ ]:
import json
import requests
import sys
import oasis
import time
from IPython.display import JSON

from arango import ArangoClient

In [ ]:
# Retrieve tmp credentials from ArangoDB Tutorial Service
login = oasis.getTempCredentials(tutorialName="Pregel", credentialProvider='https://tutorials.arangodb.cloud:8529/_db/_system/tutorialDB/tutorialDB')

# Connect to the temp database
db = oasis.connect_python_arango(login)

Requesting new temp credentials.
Temp database ready to use.


In [ ]:
print("https://{}:{}".format(login["hostname"], login["port"]))
print("Username: " + login["username"])
print("Password: " + login["password"])
print("Database: " + login["dbName"])

https://tutorials.arangodb.cloud:8529
Username: TUTf35saujsqz5b2qd7f8ixd9
Password: TUTx14u5izh08mxixjgptnnhc
Database: TUT581w65l03csy6vi6fvwisg


In [ ]:
#descarga de archivos
!wget https://github.com/adigenova/uohpmd/blob/main/data/nosql/aeropuertos.csv.gz/?raw=true -O aeropuertos.csv.gz
!wget https://github.com/adigenova/uohpmd/blob/main/data/nosql/vuelos.csv.gz/?raw=true -O vuelos.csv.gz


--2024-11-19 20:18:02--  https://github.com/adigenova/uohpmd/blob/main/data/nosql/aeropuertos.csv.gz/?raw=true
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/adigenova/uohpmd/raw/refs/heads/main/data/nosql/aeropuertos.csv.gz [following]
--2024-11-19 20:18:03--  https://github.com/adigenova/uohpmd/raw/refs/heads/main/data/nosql/aeropuertos.csv.gz
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/adigenova/uohpmd/refs/heads/main/data/nosql/aeropuertos.csv.gz [following]
--2024-11-19 20:18:03--  https://raw.githubusercontent.com/adigenova/uohpmd/refs/heads/main/data/nosql/aeropuertos.csv.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com 

Cargamos los datos a la DB

In [ ]:
# Descomprimir y mostrar las primeras líneas de los archivos
! zcat /content/aeropuertos.csv.gz | wc -l
! zcat /content/vuelos.csv.gz | wc -l
! zcat /content/aeropuertos.csv.gz > /content/aeropuertos.csv
! zcat /content/vuelos.csv.gz > /content/vuelos.csv
! echo ":::AEROPUERTOS:::"
! head /content/aeropuertos.csv
! echo ":::VUELOS:::"
! head /content/vuelos.csv

In [ ]:
# cargamos los aeropuertos A LA db
!./tools/arangoimport  -c none --server.endpoint http+ssl://{login["hostname"]}:{login["port"]} --server.username {login["username"]} --server.database {login["dbName"]} --server.password {login["password"]} --file /content/aeropuertos.csv --collection airports --create-collection true --type csv

In [ ]:
# CARGAMOS los vuelos a la DB
! ./tools/arangoimport  -c none --server.endpoint http+ssl://{login["hostname"]}:{login["port"]} --server.username {login["username"]} --server.database {login["dbName"]} --server.password {login["password"]} --file /content/vuelos.csv --collection flights --create-collection true --type csv --create-collection-type edge

In [ ]:
aql = db.aql

Consultas

In [ ]:
# consulta 1
query = """
FOR flight IN flights
    FILTER flight.TailNum == "N802AE"
    SORT flight.Distance DESC
    LIMIT 10
    RETURN { origen: flight._from, destino: flight._to, distancia: flight.Distance }
"""
results = aql.execute(query)
for result in results:
    print(result)


In [ ]:
# consulta 2
query = """
FOR flight IN flights
    FILTER flight.TailNum == "N802AE"
    COLLECT AGGREGATE minDist = MIN(flight.Distance), maxDist = MAX(flight.Distance),
                       avgDist = AVG(flight.Distance), stdDevDist = STDDEV(flight.Distance)
    RETURN { minima: minDist, maxima: maxDist, promedio: avgDist, desviacion: stdDevDist }
"""
result = aql.execute(query).next()
print(result)


{'minima': 89, 'maxima': 1021, 'promedio': 565.0298507462686, 'desviacion': 302.7851792617577}


In [ ]:
# consulta 3
query = """
FOR flight IN flights
    FILTER flight.TailNum == "N802AE"
    RETURN DISTINCT { origen: flight._from, destino: flight._to }
"""
results = aql.execute(query)
for result in results:
    print(result)


In [ ]:
# consulta 4
query = """
FOR flight IN flights
    COLLECT TailNum = flight.TailNum WITH COUNT INTO numflights
    SORT numflights DESC
    LIMIT 5
    RETURN { avion: TailNum, cantidad_vuelos: numflights }
"""
results = aql.execute(query)
for result in results:
    print(result)


In [ ]:
# consulta 5
query = """
FOR flight IN flights
    COLLECT tailNum = flight.TailNum WITH COUNT INTO numFlights
    SORT numFlights DESC
    LIMIT 1
    RETURN { tailNum, total_flights: numFlights }
"""
results = aql.execute(query)
results_data = list(results)
top_tailnum = results_data[0]['tailNum']
print(f"Avion con mas vuelos: {top_tailnum}")

query_graph = f"""
WITH airports, flights
FOR flight IN flights
    FILTER flight.TailNum == "{top_tailnum}"
    RETURN {{ origin: flight._from, destination: flight._to }}
"""
graph_data = list(aql.execute(query_graph))
print(graph_data)

import networkx as nx
import matplotlib.pyplot as plt

# cracion del grafo
G = nx.DiGraph()  # grafo dirigido

for flight in graph_data:
    G.add_edge(flight['origin'], flight['destination'])

# dibujo del grafo
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_size=600, node_color="yellow", font_size=10, font_weight="bold")
plt.title(f"Grafo de vuelos del avion con mas vuelos ({top_tailnum})")
plt.show()


In [ ]:
# consulta 6
query = """
WITH airports, flights
FOR v, e, p IN OUTBOUND "airports/ATL" flights
    RETURN { destination: v._key, path: p }
"""
# Ejecutar la consulta
results = list(aql.execute(query))
print(results)

import networkx as nx
import matplotlib.pyplot as plt

# Crear el grafo
G = nx.DiGraph()

# Agregar nodos y aristas al grafo
for connection in results:
    destination = connection['destination']
    path = connection['path']
    for edge in path['edges']:
        G.add_edge(edge['_from'], edge['_to'])

# dibujo del grafo
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=600, node_color="yellow", font_size=10, font_weight="bold")
plt.title("Conexiones OUTBOUND desde ATL")
plt.show()

In [ ]:
# consulta 7
query = """
WITH airports, flights
FOR v, e, p IN OUTBOUND "airports/ATL" flights
    RETURN { destino: e._to, escalas: LENGTH(p.edges), distancia_total: SUM(p.edges[*].Distance) }
"""
results = aql.execute(query)
for result in results:
    print(result)


Parte 4.8

In [ ]:
import gzip
import shutil
import sqlite3
import pandas as pd

aeropuertos = 'aeropuertos.csv.gz'
vuelos= 'vuelos.csv.gz'

with gzip.open(aeropuertos, 'rb') as f_in:
    with open('aeropuertos.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

with gzip.open(vuelos, 'rb') as f_in:
    with open('vuelos.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

# crear la conexión con SQLite
conn = sqlite3.connect('AeropuertosVuelos.db')

# carga de los datos de los CSV en pandas
aeropuertos = pd.read_csv('aeropuertos.csv')
vuelos = pd.read_csv('vuelos.csv')

# escribir los datos en SQLite
aeropuertos.to_sql('aeropuertos', conn, if_exists='replace', index=False)
vuelos.to_sql('vuelos', conn, if_exists='replace', index=False)


consulta 1 en SQLite y NoSQL

In [ ]:
%%time
# consulta con SQLite
sql_query_1 = """
SELECT _from, _to, Distance
FROM vuelos
WHERE TailNum = "N802AE"
ORDER BY Distance DESC
LIMIT 10;
"""
result_sql_1 = pd.read_sql_query(sql_query_1, conn)
print(result_sql_1)



In [ ]:
%%time
# consulta con NoSQL (ArangoDB)
query_1 = """
WITH airports, flights
FOR flight IN flights
    FILTER flight.TailNum == "N802AE"
    SORT flight.Distance DESC
    LIMIT 10
    RETURN { origin: flight._from, destination: flight._to, distance: flight.Distance }
"""
result_arangodb_1 = aql.execute(query_1)
for res in result_arangodb_1:
    print(res)

Consulta 4 con SQLite y NoSQL

In [ ]:
%%time
# consulta con SQLite
sql_query_4 = """
SELECT TailNum AS flight, COUNT(*) AS total_flights
FROM vuelos
GROUP BY TailNum
ORDER BY total_flights DESC
LIMIT 5;
"""
result_sql_4 = pd.read_sql_query(sql_query_4, conn)
print(result_sql_4)

In [ ]:
%%time
# consulta con NoSQL (ArangoDB)
query_4 = """
FOR flight IN flights
    COLLECT tailNum = flight.TailNum WITH COUNT INTO numFlights
    SORT numFlights DESC
    LIMIT 5
    RETURN { flight: tailNum, total_flights: numFlights }
"""
result_arangodb_4 = aql.execute(query_4)
for res in result_arangodb_4:
    print(res)
